# Module 0: Prerequisites & Environment Setup

---

## Welcome to the Amazon Bedrock AgentCore Workshop!

In this workshop, you will build **Aria** -- a production-grade AI assistant powered by **Amazon Bedrock AgentCore**. Over the course of 9 modules, you will go from zero to a fully deployed, secure, observable AI agent.

![Aria AI Chat](../shared/img/aria-home.png)

### What you will build

Aria is not a toy demo. By the end of this workshop, your assistant will:

- **Run in AgentCore Runtime** -- deployed as a managed, scalable agent endpoint
- **Use Code Interpreter & Browser tools** -- to execute code and browse the web on behalf of users
- **Remember conversations** -- with AgentCore Memory for persistent, cross-session context
- **Expose a secure API** -- through AgentCore Gateway with Identity-based authentication
- **Enforce fine-grained policies** -- using Cedar policies via AgentCore Policy
- **Emit traces and metrics** -- through AgentCore Observability
- **Pass quality checks** -- validated by AgentCore Evaluations

![Aria AI Demo](../shared/img/aria-demo.png)

### Workshop modules

| Module | Topic |
|--------|-------|
| **00** | Prerequisites & Environment Setup (this module) |
| **01** | Introduction to Amazon Bedrock AgentCore |
| **02** | Deploy Your First Agent to Runtime |
| **03** | Add Code Interpreter & Browser Tools |
| **04** | Add Persistent Memory |
| **05** | Connect Gateway & Identity |
| **06** | Enforce Cedar Policies |
| **07** | Observability & Evaluations |
| **08** | Full Production Deployment |

Let's start by verifying that your environment is ready.

---

## 1. Environment Verification

We need to confirm that the required tools are installed and that your AWS credentials are configured correctly.

### 1.1 Python version

This workshop requires **Python 3.12**.

In [ ]:
!python3 --version

### 1.2 AWS CLI

The AWS CLI is used for various operations throughout the workshop.

In [ ]:
!aws --version

### 1.3 AWS CDK

The AWS CDK is used in the deployment modules.

In [ ]:
!cdk --version

### 1.4 AWS Credentials

Verify that your AWS credentials are valid and that you can make API calls.

In [ ]:
import sys
sys.path.insert(0, '..')

import boto3

sts = boto3.client('sts')
identity = sts.get_caller_identity()

print(f"Account:  {identity['Account']}")
print(f"Arn:      {identity['Arn']}")
print(f"UserId:   {identity['UserId']}")
print("\nAWS credentials are valid.")

---

## 1.5 X-Ray Transaction Search Check

The prerequisites CloudFormation stack enables **X-Ray Transaction Search** by default. This is an account-level setting (only one configuration is allowed per account per region), and it is required for the observability module later in the workshop.

If your account already has Transaction Search enabled, the stack deployment will fail unless you set the `EnableTransactionSearch` parameter to `false`. Run the cell below to check your account's current status **before deploying the stack**.

In [ ]:
import boto3
from botocore.exceptions import ClientError

region = "us-east-1"
xray = boto3.client("xray", region_name=region)

try:
    response = xray.get_indexing_rules()
    indexing_rules = response.get("IndexingRules", [])

    enabled = False
    for rule in indexing_rules:
        probabilistic = rule.get("Rule", {}).get("Probabilistic", {})
        if probabilistic.get("DesiredSamplingPercentage", 0) > 0:
            enabled = True
            pct = probabilistic["DesiredSamplingPercentage"]
            break

    if enabled:
        print(f"✅ X-Ray Transaction Search is ENABLED (indexing {pct}% of traces).")
        print()
        print("   Your account already has Transaction Search configured.")
        print("   When deploying the prerequisites stack, add this parameter")
        print("   to skip creating a duplicate configuration:")
        print()
        print("   aws cloudformation deploy \\")
        print("     --template-file infrastructure/prerequisites.yaml \\")
        print("     --stack-name agentcore-workshop-prerequisites \\")
        print("     --capabilities CAPABILITY_NAMED_IAM \\")
        print("     --parameter-overrides EnableTransactionSearch=false \\")
        print("     --region us-east-1")
    else:
        print("ℹ️  X-Ray Transaction Search is NOT currently enabled in this account/region.")
        print()
        print("   The prerequisites stack will enable it automatically (default behavior).")
        print("   No extra parameters needed -- just deploy with the standard command:")
        print()
        print("   aws cloudformation deploy \\")
        print("     --template-file infrastructure/prerequisites.yaml \\")
        print("     --stack-name agentcore-workshop-prerequisites \\")
        print("     --capabilities CAPABILITY_NAMED_IAM \\")
        print("     --region us-east-1")

except ClientError as e:
    print(f"⚠️  Could not check Transaction Search status: {e}")
    print()
    print("   If the stack deployment fails on the TransactionSearchConfig resource,")
    print("   redeploy with --parameter-overrides EnableTransactionSearch=false")
except Exception as e:
    print(f"⚠️  Unexpected error checking Transaction Search: {e}")
    print()
    print("   If the stack deployment fails on the TransactionSearchConfig resource,")
    print("   redeploy with --parameter-overrides EnableTransactionSearch=false")

---

## 2. CloudFormation Stack Verification

If you followed the README instructions, you deployed the prerequisites CloudFormation stack using `aws cloudformation deploy`. Let's verify that the stack completed successfully and that all required outputs are available.

In [ ]:
import sys
sys.path.insert(0, '..')

from shared.progress import check_prerequisites

check_prerequisites()

---

## 3. What Was Provisioned

The CloudFormation stack created the following resources that Aria will use throughout the workshop:

### Authentication & Identity

- **Amazon Cognito User Pool** -- Manages user accounts for Aria's end users
- **Cognito App Client** -- Allows Aria's frontend to authenticate users
- **Cognito Domain** -- Provides hosted UI endpoints for sign-in flows

### Data & APIs

- **Amazon DynamoDB Table** -- Stores tasks that Aria can create, read, update, and delete
- **AWS Lambda Function** -- Implements the Task API business logic
- **Amazon API Gateway REST API** -- Exposes the Task API as a secure HTTP endpoint

### Storage

- **Amazon S3 Bucket** -- Stores artifacts, logs, and other files generated during the workshop

### IAM Roles

- **Runtime Execution Role** -- Grants Aria's agent the permissions it needs when running in AgentCore Runtime
- **Gateway Execution Role** -- Grants AgentCore Gateway the permissions to invoke and manage agent endpoints

### Observability

- **X-Ray Transaction Search** -- Enables distributed trace indexing so you can search and analyze agent traces in the observability module. This is an account-level setting that indexes 100% of traces and takes approximately 10 minutes to become active. If your account already had Transaction Search enabled, this resource was skipped (controlled by the `EnableTransactionSearch` stack parameter).
- **CloudWatch Logs Resource Policy** -- Grants X-Ray permission to write trace spans to CloudWatch Logs (`aws/spans` log group)

These resources form the backbone of Aria's environment. In the modules ahead, you will connect Aria to each of them.

---

## 4. Mark Module Complete

Everything checks out! Let's record your progress.

In [ ]:
import sys
sys.path.insert(0, '..')

from shared.progress import show

show("00")

---

**Next up: [Module 1 -- Introduction to Amazon Bedrock AgentCore](../01-introduction/notebook.ipynb)**